In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models # Keras 레이어 / 모델 유틸

# Residual Block(ResNet의 기본 블록)을 만드는 함수
def residual_block(inputs, filters, downsample=False):
    x = inputs
    if downsample:
        strides = 2
    else:
        strides = 1

    # 첫 번째 합성곱 층
    x = layers.Conv2D(filters, kernel_size=3, strides=strides, padding='same')(x)   # 필요시 다운샘플링
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # 두 번째 합성곱 층
    x = layers.Conv2D(filters, kernel_size=3, strides=1, padding='same')(x)         # 크기 유지
    x = layers.BatchNormalization()(x)

    # 스킵 연결
    if downsample:  # 입력과 출력의 shape을 맞춰줘야 하는 경우
        inputs = layers.Conv2D(filters, kernel_size=1, strides=2, padding='same')(inputs)   # 채널/크기 정렬
        inputs = layers.BatchNormalization()(inputs)

    x = layers.add([inputs, x])
    x = layers.ReLU()(x)
    return x

In [ ]:
from tensorflow.keras.datasets import cifar10

# 데이터셋 로드 및 전처리
(train_images, train_labels), (test_images, test_labels) = cifar10.load_data()

# 픽셀 값 정규화
train_images, test_images = train_images / 255.0, test_images / 255.0

In [ ]:
# 입력층
inputs = layers.Input(shape=(32, 32, 3))

# 초기 합성곱 층
x = layers.Conv2D(64, kernel_size=7, strides=2, padding='same')(inputs)
x = layers.BatchNormalization()(x)
x = layers.ReLU()(x)
x = layers.MaxPooling2D(pool_size=3, strides=2, padding='same')(x)

# Residual Block 추가
x = residual_block(x, filters=64)
x = residual_block(x, filters=64)

x = residual_block(x, filters=128, downsample=True)
x = residual_block(x, filters=128)

x = residual_block(x, filters=256, downsample=True)
x = residual_block(x, filters=256)

x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(10, activation='softmax')(x)

# 모델 생성
model = models.Model(inputs=inputs, outputs=outputs)

# 모델 요약 출력
model.summary()

In [ ]:
# 모델 컴파일
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 모델 학습
model.fit(train_images, train_labels, epochs=10, batch_size=64, validation_split=0.1)

In [ ]:
# 모델 평가
test_loss, test_acc = model.evaluate(test_images, test_labels)
print('테스트 정확도:', test_acc)